# Week 1: Python & NumPy Foundations
## Notebook 07: Memory Strides, C vs. Fortran Layout & Cache Locality

### 1. Conceptual Intuition & Real-World Use Case
Computer RAM is a flat, 1-dimensional array of bytes. However, Machine Learning algorithms operate on 2D matrices, 3D image volumes, and 4D tensors.

How does NumPy map a multi-dimensional index $(i, j, k)$ to a 1D physical RAM address?
The answer is **Memory Strides**:
- A stride tuple tells NumPy exactly how many **bytes** it must step forward in RAM to reach the next element along each dimension.
- Understanding strides is the difference between an algorithm that runs in 2 milliseconds (cache-friendly) versus 200 milliseconds (cache-thrashing).

---

### 2. Mathematical Foundations: Strides & Row vs. Column Major
For an array of data type size $S$ bytes (e.g., float64 = 8 bytes) with shape $(D_0, D_1)$:

1. **C-Contiguous (Row-Major - Default in C, Python):**
   Rows are stored sequentially in memory.
   $$\text{Address}(i, j) = \text{Base} + (i \cdot D_1 + j) \times S$$
   Strides: $(D_1 \times S, S)$. Stepping to the next row jumps $D_1 \times S$ bytes; stepping to the next column jumps $S$ bytes.

2. **Fortran-Contiguous (Column-Major - Default in MATLAB, Fortran, R):**
   Columns are stored sequentially in memory.
   $$\text{Address}(i, j) = \text{Base} + (i + j \cdot D_0) \times S$$
   Strides: $(S, D_0 \times S)$.

3. **Zero-Copy Views:**
   Operations like `arr.T` (transpose) or `arr.reshape()` do NOT copy array data! They simply alter the shape and stride metadata.
   When strides become non-contiguous, CPU hardware prefetchers cannot load consecutive cache lines, destroying performance.

In [ ]:
import time
import numpy as np

# Step 1: Inspect Memory Strides of C-Contiguous Array
# 3 rows, 4 columns of float64 (8 bytes each)
arr_c = np.arange(12, dtype=np.float64).reshape(3, 4)

print("Array Shape:", arr_c.shape)
print("Item Size (bytes):", arr_c.itemsize)
print("Memory Strides (bytes to advance per dimension):", arr_c.strides)
print("Flags C-Contiguous:", arr_c.flags['C_CONTIGUOUS'])
print("Flags F-Contiguous:", arr_c.flags['F_CONTIGUOUS'])

# To move to the next row (dimension 0), jump 4 elements * 8 bytes = 32 bytes!
# To move to the next column (dimension 1), jump 1 element * 8 bytes = 8 bytes!

# Step 2: Transposition is a Zero-Copy Metadata Mutation
arr_transposed = arr_c.T
print("\n--- Transposed Array (No Data Copy!) ---")
print("Transposed Shape:", arr_transposed.shape)
print("Transposed Strides:", arr_transposed.strides)
print("Shares Memory with Original:", np.may_share_memory(arr_c, arr_transposed))
print("Flags C-Contiguous:", arr_transposed.flags['C_CONTIGUOUS'])
print("Flags F-Contiguous:", arr_transposed.flags['F_CONTIGUOUS'])

# Step 3: Cache Locality Performance Benchmark
# CPU Cache Lines load 64 bytes at a time.
# Traversing along contiguous strides hits L1/L2 cache; traversing non-contiguous strides causes cache misses!
N = 5000
large_c = np.ones((N, N), dtype=np.float64, order='C')

# Benchmark 1: Row-wise traversal (Sequential in RAM for C-order)
start = time.perf_counter()
sum_rows = np.sum(large_c, axis=1) # iterates along contiguous rows
time_row = time.perf_counter() - start

# Benchmark 2: Column-wise traversal (Jumping across RAM for C-order)
start = time.perf_counter()
sum_cols = np.sum(large_c, axis=0) # iterates across non-contiguous strides
time_col = time.perf_counter() - start

print(f"\n--- Benchmark on {N}x{N} Matrix ({large_c.nbytes / (1024**2):.1f} MB) ---")
print(f"Row-wise sum (Contiguous cache access):     {time_row * 1000:.2f} ms")
print(f"Column-wise sum (Non-contiguous jumping):   {time_col * 1000:.2f} ms")
print(f"Speedup from CPU Cache Alignment:           {time_col / max(1e-6, time_row):.2f}x faster!")

### 3. Step-by-Step Code Walkthrough
1. **Stride Accounting:** The 32-byte row stride reflects 4 elements $\times$ 8 bytes.
2. **Zero-Copy Transposition:** Transposing swapped the strides from $(32, 8)$ to $(8, 32)$ without copying a single byte of memory.
3. **Cache Benchmark:** Summing along contiguous rows is dramatically faster because modern CPU hardware prefetchers load contiguous 64-byte cache lines into L1 cache ahead of time.

---

### 4. Industry Applications & Trade-offs
- **Convolutional Neural Networks:** PyTorch's `as_strided()` implements sliding window 2D image patches with zero memory duplication.
- **Interoperability:** Passing NumPy arrays to C/CUDA libraries requires `np.ascontiguousarray()` to guarantee expected pointer arithmetic.

---

### 5. Top Interview Questions & Answers
**Q1: What does `np.ascontiguousarray(x)` do and when is it necessary?**
> **Answer:** If an array has been sliced or transposed with negative or non-standard strides, `np.ascontiguousarray()` allocates a new contiguous block in memory and copies the data into C-order. This is mandatory before passing NumPy arrays to C extensions, Cython, or PyTorch CUDA kernels that expect contiguous memory pointers.

**Q2: What is the difference between an array view and an array copy?**
> **Answer:** A view shares the underlying data buffer with the parent array, altering only shape, strides, and offset metadata. Mutating elements in a view mutates the original array. A copy allocates independent memory. Slicing (`arr[1:5]`) creates a view, whereas fancy indexing (`arr[[0, 2, 4]]`) always creates a copy.